## First all base models are loaded Hints_IB,Rule based models and MT5 small. 
Here using gen_all_model_op function generate a dict containing original unnorm sentence and op of each model

In [ ]:
import torch
from torch.utils.data import DataLoader,Dataset
from torch.nn.utils.rnn import pad_sequence
import json
import random
from transformers import MT5ForConditionalGeneration
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from TB_RB import normalize_sent
import ast
import os
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, Trainer, TrainingArguments,
    TrainerCallback
)
import evaluate
import numpy as np
from safetensors.torch import load_file

from indic_numtowords import num2words
import re
import json
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoModel, BertModel, AutoConfig, TrainingArguments, Trainer
import json
import torch
from typing import List, Dict, Optional



# =======================
# CONSTANTS & MAPPINGS
# =======================

class_to_id = {
    "O": 0,
    "B-DATE": 1, "I-DATE": 2,
    "B-CARDINAL": 3, "I-CARDINAL": 4,
    "B-FRACTION": 5, "I-FRACTION": 6,
    "B-MONEY": 7, "I-MONEY": 8,
    "B-TELEPHONE": 9, "I-TELEPHONE": 10,
    "B-MEASURE": 11, "I-MEASURE": 12,
    "B-TIME": 13, "I-TIME": 14,
    "B-DECIMAL": 15, "I-DECIMAL": 16,
}

id_to_class = {v: k for k, v in class_to_id.items()}

language_mapping = {"en": 0, "hi": 1, "te": 2, "kan": 3}

entity_type_mapping = {
    "CARDINAL": 0,
    "DATE": 1,
    "FRACTION": 2,
    "MONEY": 3,
    "TELEPHONE": 4,
    "MEASURE": 5,
    "TIME": 6,
    "DECIMAL": 7
}

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# =======================
# LOAD TOKENIZER & NER MODEL
# =======================

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/IndicBERTv2-MLM-Sam-TLM")
label_list = list(class_to_id.keys())

# Define BAD_TOKENS based on the loaded tokenizer
BAD_TOKENS = tokenizer.all_special_tokens

ner_model = AutoModelForTokenClassification.from_pretrained(
    "ai4bharat/IndicBERTv2-MLM-Sam-TLM",
    num_labels=len(label_list)
)

# Load NER weights
weights_path = "/content/model.safetensors"
state_dict = load_file(weights_path)
ner_model.load_state_dict(state_dict)
ner_model = ner_model.to(device)
ner_model.eval()
print("✅ Loaded NER model with safetensors weights successfully!")

# =======================
# LOAD MT5 MODEL (NEW ADDITION)
# =======================

print("🔄 Loading MT5 model...")
mt5_model_name = "google/mt5-small"
tokenizer_mt5 = AutoTokenizer.from_pretrained(mt5_model_name)
mt5_model = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

# Load MT5 checkpoint if available
mt5_checkpoint_path = "/content/checkpoint-step110000.pt"
if os.path.exists(mt5_checkpoint_path):
    mt5_model.load_state_dict(torch.load(mt5_checkpoint_path, map_location=device))
    print("✅ Loaded MT5 checkpoint successfully!")
else:
    print("⚠️ MT5 checkpoint not found, using base model")

mt5_model = mt5_model.to(device)
mt5_model.eval()
print("✅ MT5 model loaded successfully!")

# =======================
# HELPER: get_hint_from_span
# =======================

def get_hint_from_span(span: str, lang="en"):
    """
    Convert numeric spans to spoken word hints.
    """
    span = span.strip()
    if not span:
        return span, False

    lang_map = {'hi': 'hi', 'en': 'en', 'te': 'te', 'kan': 'kn'}
    target_lang = lang_map.get(lang, 'en')

    try:
        # Pure number
        if span.isdigit():
            return num2words(int(span), lang=target_lang), False

        # Decimal numbers
        if re.fullmatch(r"\d+\.\d+", span):
            integer, fractional = span.split(".")
            int_part = num2words(int(integer), lang=target_lang)
            frac_part = " ".join(num2words(int(d), lang=target_lang) for d in fractional)
            return f"{int_part} point {frac_part}", False

        # Fractions (e.g., "2/3")
        if re.fullmatch(r"\d+/\d+", span):
            numerator, denominator = span.split("/")
            num_words = num2words(int(numerator), lang=target_lang)
            denom_words = num2words(int(denominator), lang=target_lang)

            if target_lang == 'en':
                denom_suffix = ""
                if denominator == "2":
                    denom_suffix = "half"
                elif denominator == "3":
                    denom_suffix = "third"
                elif denominator == "4":
                    denom_suffix = "quarter"
                else:
                    denom_suffix = f"{denom_words}th"

                if numerator == "1":
                    return f"one {denom_suffix}", False
                else:
                    return f"{num_words} {denom_suffix}s", False
            else:
                return f"{num_words} by {denom_words}", False

        # Handle tokens with special characters
        tokens = re.findall(r'(\d+|\D+)', span)
        hint_tokens = []

        for token in tokens:
            if token.isdigit():
                try:
                    num_words = num2words(int(token), lang=target_lang)
                    hint_tokens.append(num_words)
                except:
                    hint_tokens.append(token)
            else:
                if '#' in token:
                    sub_tokens = re.findall(r'(\d+|#+)', token)
                    sub_hints = []
                    for sub in sub_tokens:
                        if sub.isdigit():
                            try:
                                sub_hints.append(num2words(int(sub), lang=target_lang))
                            except:
                                sub_hints.append(sub)
                        else:
                            sub_hints.append(sub)
                    hint_tokens.append(''.join(sub_hints))
                else:
                    hint_tokens.append(token)

        hint_text = ''.join(hint_tokens)
        hint_text = re.sub(r'\s+([/+\-:])', r'\1', hint_text)
        hint_text = re.sub(r'([/+\-:])\s+', r'\1', hint_text)

        if hint_text == span:
            return span, True
        else:
            return hint_text, False

    except Exception as e:
        print(f"Error converting '{span}': {e}")
        return span, True

# =======================
# BERT DECODER MODEL
# =======================

class BertDecoderOnlyLM(torch.nn.Module):
    def __init__(
        self,
        vocab_size,
        num_languages,
        num_entity_types,
        tokenizer,
        model_name="ai4bharat/IndicBERTv2-MLM-Sam-TLM",
        max_len=512,
        sep_token_id=None,
        padding_token_id=None,
    ):
        super(BertDecoderOnlyLM, self).__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.tokenizer = tokenizer

        self.config = AutoConfig.from_pretrained(model_name)
        self.config.is_decoder = True
        self.config.add_cross_attention = False
        self.hidden_size = self.config.hidden_size

        self.decoder = BertModel.from_pretrained(model_name, config=self.config)
        self.decoder.resize_token_embeddings(len(self.tokenizer))

        self.ln_f = nn.LayerNorm(self.hidden_size)
        self.head = nn.Linear(self.hidden_size, vocab_size, bias=False)
        self.head.weight = self.decoder.embeddings.word_embeddings.weight

        self.lang_embedding = nn.Embedding(num_languages, self.hidden_size)
        self.entity_embedding = nn.Embedding(num_entity_types, self.hidden_size)

        self.sep_token_id = sep_token_id if sep_token_id is not None else 105
        self.padding_token_id = padding_token_id if padding_token_id is not None else 0

        self.loss_fct = nn.CrossEntropyLoss(ignore_index=self.padding_token_id)

    def forward(
        self,
        unnorm_span_ids,
        hint_input_ids,
        decoder_input_ids,
        entity_ids,
        language_ids,
        labels=None,
        **kwargs
    ):
        batch_size = unnorm_span_ids.size(0)

        lang_emb = self.lang_embedding(language_ids)
        if len(lang_emb.shape) == 2:
            lang_emb = lang_emb.unsqueeze(1)

        ent_emb = self.entity_embedding(entity_ids)
        if len(ent_emb.shape) == 2:
            ent_emb = ent_emb.unsqueeze(1)

        hint_emb = self.decoder.embeddings(hint_input_ids)
        span_emb = self.decoder.embeddings(unnorm_span_ids)
        dec_emb = self.decoder.embeddings(decoder_input_ids)

        prompt_seq = torch.cat([lang_emb, ent_emb, hint_emb], dim=1)
        full_input = torch.cat([prompt_seq, span_emb, dec_emb], dim=1)

        decoder_output = self.decoder(inputs_embeds=full_input)[0]

        x_norm = self.ln_f(decoder_output)
        logits_all = self.head(x_norm)

        L_prompt = prompt_seq.size(1)
        L_span = span_emb.size(1)
        context_len = L_prompt + L_span

        logits = logits_all[:, context_len:, :]

        if labels is not None:
            target = labels[:, 1:].contiguous()
            shifted_logits = logits[:, :-1, :]
            loss = self.loss_fct(shifted_logits.reshape(-1, shifted_logits.size(-1)), target.reshape(-1))
            return {"loss": loss, "logits": logits}

        return {"logits": logits}

# =======================
# LOAD DECODER WEIGHTS
# =======================

ib_decoder = BertDecoderOnlyLM(
    vocab_size=len(tokenizer),
    num_languages=len(language_mapping),
    num_entity_types=len(entity_type_mapping),
    tokenizer=tokenizer,
    model_name="ai4bharat/IndicBERTv2-MLM-Sam-TLM"
)

decoder_weights = torch.load(
    "/content/IB_decoder_weights_updated.pt",
    map_location=device
)
ib_decoder.load_state_dict(decoder_weights)
ib_decoder = ib_decoder.to(device)
ib_decoder.eval()
print("✅ Loaded IndicBERT decoder weights successfully!")

# =======================
# NER FUNCTION
# =======================

def ner_single(text: str):
    """
    Perform NER on a single text and return list of entities.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
        return_offsets_mapping=True
    )
    offset_mapping = inputs.pop("offset_mapping")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = ner_model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2).cpu().numpy()
    input_ids = inputs["input_ids"].cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    labels = [id_to_class[p] for p in predictions[0]]
    offsets = offset_mapping[0].tolist()

    entities = []
    current = None

    for (token, label, (start, end)) in zip(tokens, labels, offsets):
        if token in BAD_TOKENS:
            continue

        if label.startswith("B-"):
            if current:
                entities.append(current)
            current = {
                "text": token.replace("##", ""),
                "start": start,
                "end": end,
                "type": label[2:]
            }
        elif label.startswith("I-") and current and current["type"] == label[2:]:
            current["text"] += token.replace("##", "")
            current["end"] = end
        else:
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    return entities

# =======================
# DECODE SPANS FUNCTION (INDICBERT)
# =======================

def decode_single_span(span: str, entity_type: str, language="kan", max_new_tokens=20):
    """
    Decode a single unnormalized span using the IndicBERT decoder.
    """
    ib_decoder.eval()

    # Get hint
    hint_text = get_hint_from_span(span, lang=language)[0]

    # Tokenize
    hint_input_ids = tokenizer(
        hint_text,
        return_tensors="pt",
        truncation=True,
        max_length=16
    ).input_ids.to(device)

    unnorm_span_ids = tokenizer(
        span,
        return_tensors="pt",
        truncation=True,
        max_length=32
    ).input_ids.to(device)

    decoder_input_ids = torch.full(
        (1, 1),
        tokenizer.cls_token_id,
        dtype=torch.long,
        device=device
    )

    entity_ids = torch.tensor(
        [entity_type_mapping[entity_type]],
        dtype=torch.long,
        device=device
    )
    language_ids = torch.tensor(
        [language_mapping[language]],
        dtype=torch.long,
        device=device
    )

    generated = decoder_input_ids.clone()
    finished = False
    token_count = 0

    with torch.no_grad():
        while not finished and token_count < max_new_tokens:
            outputs = ib_decoder(
                unnorm_span_ids=unnorm_span_ids,
                hint_input_ids=hint_input_ids,
                decoder_input_ids=generated,
                entity_ids=entity_ids,
                language_ids=language_ids
            )

            logits = outputs["logits"]
            next_token_id = torch.argmax(logits[:, -1, :], dim=-1)

            generated = torch.cat([generated, next_token_id.unsqueeze(1)], dim=1)
            token_count += 1

            token_str = tokenizer.convert_ids_to_tokens(next_token_id[0].item())
            if token_str in BAD_TOKENS:
                finished = True

    # Detokenize
    def wordpiece_detokenize(tokens):
        out = []
        for tok in tokens:
            if tok.startswith("##") and len(out) > 0:
                out[-1] = out[-1] + tok[2:]
            else:
                out.append(tok)
        return " ".join(out)

    tokens = tokenizer.convert_ids_to_tokens(generated[0][1:])  # drop CLS
    tokens = [t for t in tokens if t not in BAD_TOKENS]
    text = wordpiece_detokenize(tokens)

    # CARDINAL: remove anything after decimal marker
    if entity_type == "CARDINAL":
        decimal_markers = ["ಪಾಯಿಂಟ್", "point", "dot", "दशमलव", "decimal"]
        for marker in decimal_markers:
            if marker in text:
                text = text.split(marker)[0].strip()
                break

    return text

# =======================
# MT5 GENERATION FUNCTION (NEW ADDITION)
# =======================

def generate_mt5(text: str, max_new_tokens=128):
    """
    Generate normalized text using MT5 model for a single input.
    """
    # Tokenize input
    inputs = tokenizer_mt5(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Generate output
    with torch.no_grad():
        output_ids = mt5_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            do_sample=False
        )

    # Decode output
    output_text = tokenizer_mt5.decode(output_ids[0], skip_special_tokens=True)

    return output_text

# =======================
# MAIN NORMALIZATION FUNCTION (INDICBERT ONLY - KEPT AS IS)
# =======================

def normalize_sentence(text: str, language="kan"):
    """
    Main function to normalize a single sentence using only IndicBERT.
    (KEPT EXACTLY AS ORIGINAL)

    Args:
        text (str): Input unnormalized sentence
        language (str): Language code ('kan', 'hi', 'te', 'en')

    Returns:
        dict: {
            'original': original text,
            'normalized': normalized text,
            'entities': list of detected entities with their normalizations
        }
    """
    # Step 1: NER
    entities = ner_single(text)

    if not entities:
        return {
            'original': text,
            'normalized': text,
            'entities': []
        }

    # Step 2: Normalize each entity span
    normalized_text = text
    offset_shift = 0
    normalized_entities = []

    for ent in entities:
        span = ent["text"]
        entity_type = ent["type"]
        start = ent["start"] + offset_shift
        end = ent["end"] + offset_shift

        # Decode the span
        norm_span = decode_single_span(span, entity_type, language=language)

        # Apply normalization to text
        normalized_text = normalized_text[:start] + norm_span + normalized_text[end:]
        offset_shift += len(norm_span) - (end - start)

        normalized_entities.append({
            'original_span': span,
            'normalized_span': norm_span,
            'type': entity_type,
            'position': (start, end)
        })

    return {
        'original': text,
        'normalized': normalized_text,
        'entities': normalized_entities
    }

In [9]:
sent="मेरी लंबाई 175 cm है ​​और मेरे बैंक अकाउंट में $ 500 हैं।"
r=normalize_sentence(sent,language="hi")
r

{'original': 'मेरी लंबाई 175 cm है \u200b\u200bऔर मेरे बैंक अकाउंट में $ 500 हैं।',
 'normalized': 'मेरी लंबाई एक सौ पचहत्तर सेंटीमीटर है \u200b\u200bऔर मेरे बैंक अकाउंट में पांच सौ हैं।',
 'entities': [{'original_span': '175cm',
   'normalized_span': 'एक सौ पचहत्तर सेंटीमीटर',
   'type': 'MEASURE',
   'position': (11, 17)},
  {'original_span': '$500',
   'normalized_span': 'पांच सौ',
   'type': 'MONEY',
   'position': (64, 69)}]}

In [11]:
from RB_kan import normalize_sent_kan
def gen_all_model_ops(sent,language):
  hints_ib_op=normalize_sentence(sent,language=language)
  rb_op=None
  if language=="hindi":
   rb_op=normalize_sent(sent)
  elif language=="kan":
    rb_op=normalize_sent_kan(sent)
  mt5_op=generate_mt5(sent)
  return {"hints_ib":hints_ib_op,"mt5_op":mt5_op,"rb_op":rb_op,"unnorm":sent}




## Below each dynamic span detector is loaded:


In [20]:
from transformers import  MT5EncoderModel
class MT5MultiInputClassifier(nn.Module):
    def __init__(self, mt5_model_name='google/mt5-small', output_dim=3):
        super().__init__()
        # Load MT5 encoder
        self.encoder = MT5EncoderModel.from_pretrained(mt5_model_name)
        self.hidden_size = self.encoder.config.d_model  # usually 512 or 768

        # Linear layer after concatenation
        self.fc = nn.Linear(self.hidden_size * 4, output_dim)  # 4 sequences concatenated

    def forward(self, unnorm, rb_op, mt_op, h_ib_op, attention_mask=None):
        """
        Each input: (batch, seq_len)
        attention_mask is optional (if padding 0 is used, can auto-generate)
        """
        # Encode unnorm
        emb_unnorm = self.encoder(unnorm, attention_mask=(unnorm != 0).long()).last_hidden_state
        pooled_unnorm = emb_unnorm.mean(dim=1)  # simple mean pooling

        # Encode rb_op
        emb_rb = self.encoder(rb_op, attention_mask=(rb_op != 0).long()).last_hidden_state
        pooled_rb = emb_rb.mean(dim=1)

        # Encode mt_op
        emb_mt = self.encoder(mt_op, attention_mask=(mt_op != 0).long()).last_hidden_state
        pooled_mt = emb_mt.mean(dim=1)

        # Encode h_ib_op
        emb_h_ib = self.encoder(h_ib_op, attention_mask=(h_ib_op != 0).long()).last_hidden_state
        pooled_h_ib = emb_h_ib.mean(dim=1)

        # Concatenate all embeddings
        concatenated = torch.cat([pooled_unnorm, pooled_rb, pooled_mt, pooled_h_ib], dim=1)

        # Final linear layer
        logits = self.fc(concatenated)
        return logits

# =======================
# LOAD MODEL AND TOKENIZER
# =======================

def load_model(model_path, device="cpu"):
    """
    Load the trained model from checkpoint.

    Args:
        model_path (str): Path to the saved model weights
        device (str): Device to load model on

    Returns:
        model: Loaded model in evaluation mode
    """
    # Initialize model
    model = MT5MultiInputClassifier(
        mt5_model_name='google/mt5-small',
        output_dim=3
    )

    # Load weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    print(f"✅ Model loaded from: {model_path}")
    return model

def load_tokenizer():
    """Load MT5 tokenizer."""
    mt5_model_name = "google/mt5-small"
    tokenizer_mt5 = AutoTokenizer.from_pretrained(mt5_model_name)
    print("✅ Tokenizer loaded successfully!")
    return tokenizer

# =======================
# INFERENCE FUNCTION
# =======================

def select_best_output(input_dict, model, tokenizer, device="cpu"):
    """
    Select the best normalization output from 3 models (RB, MT5, Hints_IB).

    Args:
        input_dict (dict): Dictionary with 4 keys:
            - "unnorm": Original unnormalized sentence
            - "rb_op": Output from Rule-Based model
            - "mt5_op": Output from MT5 model
            - "hints_ib_op": Output from Hints_IB model
        model: Loaded MT5MultiInputClassifier model
        tokenizer: MT5 tokenizer
        device: Device to run inference on

    Returns:
        dict: {
            "unnorm": original sentence,
            "selected_output": best output text,
            "selected_model": "rb" or "mt5" or "hints_ib",
            "probabilities": [prob_rb, prob_mt5, prob_hints_ib],
            "all_outputs": {
                "rb": rb_op,
                "mt5": mt5_op,
                "hints_ib": hints_ib_op
            }
        }
    """
    model.eval()

    # Extract inputs
    unnorm = input_dict.get("unnorm", "")
    rb_op = input_dict.get("rb_op", "")
    mt5_op = input_dict.get("mt5_op", "")
    hints_ib_op = input_dict.get("hints_ib_op", "")

    # Tokenize all inputs
    with torch.no_grad():
        # Tokenize unnorm
        unnorm_tokens = tokenizer(
            unnorm,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )
        unnorm_ids = unnorm_tokens["input_ids"].to(device)

        # Tokenize rb_op
        rb_tokens = tokenizer(
            rb_op,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )
        rb_ids = rb_tokens["input_ids"].to(device)

        # Tokenize mt5_op
        mt5_tokens = tokenizer(
            mt5_op,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )
        mt5_ids = mt5_tokens["input_ids"].to(device)

        # Tokenize hints_ib_op
        hints_ib_tokens = tokenizer(
            hints_ib_op,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )
        hints_ib_ids = hints_ib_tokens["input_ids"].to(device)

        # Get model predictions
        logits = model(unnorm_ids, rb_ids, mt5_ids, hints_ib_ids)
        probs = torch.softmax(logits, dim=1)  # (1, 3)

        # Get predicted class
        pred_class = torch.argmax(probs, dim=1).item()  # 0=rb, 1=mt5, 2=hints_ib

        # Map class to model name
        model_map = {0: "rb", 1: "mt5", 2: "hints_ib"}
        selected_model = model_map[pred_class]

        # Get the selected output
        if selected_model == "rb":
            selected_output = rb_op
        elif selected_model == "mt5":
            selected_output = mt5_op
        else:  # hints_ib
            selected_output = hints_ib_op

        # Get probabilities as list
        prob_list = probs.cpu().numpy().flatten().tolist()

        # Return result
        return {
            "unnorm": unnorm,
            "selected_output": selected_output,
            "selected_model": selected_model,
            "probabilities": {
                "rb": prob_list[0],
                "mt5": prob_list[1],
                "hints_ib": prob_list[2]
            },
            "all_outputs": {
                "rb": rb_op,
                "mt5": mt5_op,
                "hints_ib": hints_ib_op
            }
        }

In [26]:
class MT5SpanSelector(nn.Module):
    """
    MT5 Encoder model that predicts the best model for normalizing each span.

    Logit indices:
        0: RB model
        1: MT5 model
        2: Hints_IB model

    Outputs logits for each model (classification, not regression).
    """

    def __init__(
        self,
        model_name: str = "google/mt5-small",
        hidden_size: int = 512,
        dropout: float = 0.1,
        num_models: int = 3
    ):
        super().__init__()

        # Load pretrained MT5 encoder
        self.encoder = MT5EncoderModel.from_pretrained(model_name)
        self.hidden_size = hidden_size

        # Projection layers
        self.span_projection = nn.Linear(hidden_size, hidden_size)

        # Separate projections for different span sources
        self.rb_proj = nn.Linear(hidden_size, hidden_size)
        self.mt5_proj = nn.Linear(hidden_size, hidden_size)
        self.hints_proj = nn.Linear(hidden_size, hidden_size)
        self.raw_proj = nn.Linear(hidden_size, hidden_size)

        # Combine all representations
        self.combine_layer = nn.Sequential(
            nn.Linear(hidden_size * 5, hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Classification head (outputs logits for RB, MT5, Hints_IB)
        self.classifier = nn.Linear(hidden_size, num_models)

        self.model_names = ['rb', 'mt5', 'hints_ib']
        self.model_to_idx = {'rb': 0, 'mt5': 1, 'hints_ib': 2}

    def encode_text(self, texts: List[str], tokenizer, max_length: int = 128) -> torch.Tensor:
        """Encode texts and return mean-pooled representations."""
        if not texts:
            return torch.zeros((0, self.hidden_size), device=self.encoder.device)

        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )

        encoded = {k: v.to(self.encoder.device) for k, v in encoded.items()}
        outputs = self.encoder(**encoded)

        # Mean pooling
        attention_mask = encoded['attention_mask'].unsqueeze(-1)
        embeddings = outputs.last_hidden_state
        masked_embeddings = embeddings * attention_mask
        summed = masked_embeddings.sum(dim=1)
        counts = attention_mask.sum(dim=1)
        mean_pooled = summed / counts

        return mean_pooled

    def forward(
        self,
        batch: Dict,
        tokenizer,
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass - predicts logits for each model.

        Returns:
            logits: [total_spans, 3] logits for RB, MT5, Hints_IB
        """
        sentences = batch['sentences']
        all_spans = batch['all_spans']
        span_to_sentence_idx = batch['span_to_sentence_idx']

        # Encode sentences
        sentence_embeddings = self.encode_text(sentences, tokenizer)

        # Prepare span texts
        rb_spans = []
        mt5_spans = []
        hints_spans = []
        raw_spans = []
        span_sentence_embeddings = []

        for span_idx, span in enumerate(all_spans):
            sent_idx = span_to_sentence_idx[span_idx]
            sent_emb = sentence_embeddings[sent_idx].unsqueeze(0)

            rb_spans.append(span.get('rb_output', '') or '')
            mt5_spans.append(span.get('mt5_output', '') or '')
            hints_spans.append(span.get('hints_ib_output', '') or '')
            raw_spans.append(span.get('raw_span', '') or '')
            span_sentence_embeddings.append(sent_emb)

        # Encode all span texts
        rb_embeddings = self.encode_text(rb_spans, tokenizer, max_length=32)
        mt5_embeddings = self.encode_text(mt5_spans, tokenizer, max_length=32)
        hints_embeddings = self.encode_text(hints_spans, tokenizer, max_length=32)
        raw_embeddings = self.encode_text(raw_spans, tokenizer, max_length=32)

        # Stack sentence embeddings
        sentence_embeddings_stack = torch.cat(span_sentence_embeddings, dim=0)

        # Apply projections
        rb_proj = self.rb_proj(rb_embeddings)
        mt5_proj = self.mt5_proj(mt5_embeddings)
        hints_proj = self.hints_proj(hints_embeddings)
        raw_proj = self.raw_proj(raw_embeddings)
        sent_proj = self.span_projection(sentence_embeddings_stack)

        # Concatenate
        combined = torch.cat([
            sent_proj, raw_proj, rb_proj, mt5_proj, hints_proj
        ], dim=1)

        # Combine features
        combined_features = self.combine_layer(combined)

        # Classification logits
        logits = self.classifier(combined_features)

        return {
            'logits': logits,  # [num_spans, 3]
        }



In [ ]:
## For sentence level dynamic model
model_path = "/content/best_model_hindi_sentence_level.pt"

    # Check if model exists
if os.path.exists(model_path):
        model = load_model(model_path)
        tokenizer = load_tokenizer()
else:
        print(" Model not found! Please train the model first.")
        print(f"Expected path: {model_path}")
        exit()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] MT5EncoderModel LOAD REPORT from: google/mt5-small
Key                                                                       | Status     |  | 
--------------------------------------------------------------------------+------------+--+-
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.q.weight     | UNEXPECTED |  | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.v.weight     | UNEXPECTED |  | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.2.DenseReluDense.wo.weight   | UNEXPECTED |  | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.{0, 1, 2}.layer_norm.weight  | UNEXPECTED |  | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.o.weight     | UNEXPECTED |  | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.2.DenseReluDense.wi_0.weight | UNEXPECTED |  | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.0.SelfAttention.k.weight     | UNEXPECTED |  | 
decoder.block.{0, 1, 2, 3, 4, 5, 6, 7}.layer.1.EncDecAttention.k.weight   | UNEXPECTED |  | 
deco

✅ Model loaded from: /content/best_model_hindi_sentence_level.pt
✅ Tokenizer loaded successfully!


In [ ]:
## FOr span level dynamic router
model_span = MT5SpanSelector(num_models=3)

    # Load weights
state_dict = torch.load("/content/best_model.pt", map_location=device)

    # Handle different checkpoint formats
if isinstance(state_dict, dict):
        if 'model_state_dict' in state_dict:
            state_dict = state_dict['model_state_dict']
        elif 'state_dict' in state_dict:
            state_dict = state_dict['state_dict']

model_span.load_state_dict(state_dict)
model_span = model_span.to(device)
model_span.eval()

In [ ]:
## Using Dynamic router which selects single best model for each sentence
hindi_test_dict = {
    "unnorm": "मेरे पास १०० रुपये हैं",
    "rb_op": "मेरे पास एक सौ रुपये हैं",
    "mt5_op": "मेरे पास एक सौ रुपये हैं",
    "hints_ib_op": "मेरे पास सौ रुपये हैं"
}
op=select_best_output(hindi_test_dict,model=model,tokenizer=tokenizer)

In [50]:
op

{'unnorm': 'मेरे पास १०० रुपये हैं',
 'selected_output': 'मेरे पास सौ रुपये हैं',
 'selected_model': 'hints_ib',
 'probabilities': {'rb': 0.35909438133239746,
  'mt5': 0.2645840644836426,
  'hints_ib': 0.3763215243816376},
 'all_outputs': {'rb': 'मेरे पास एक सौ रुपये हैं',
  'mt5': 'मेरे पास एक सौ रुपये हैं',
  'hints_ib': 'मेरे पास सौ रुपये हैं'}}

In [42]:
"""
Fixed Span Identification + Reconstruction Pipeline
=====================================================
Preserves ALL input keys and adds lm_span_selector output.
Input: a single dict with 'unnorm' and model output keys.
"""

import re
import json
import difflib
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset


# ─────────────────────────────────────────────────────────────────────────────
# Tokenizer helpers
# ─────────────────────────────────────────────────────────────────────────────

def _tokenize(text: str) -> List[Tuple[str, int]]:
    tokens = []
    for m in re.finditer(r'\S+', text):
        tokens.append((m.group(), m.start()))
    return tokens


def _tokens_only(tok_pos: List[Tuple[str, int]]) -> List[str]:
    return [t for t, _ in tok_pos]


# ─────────────────────────────────────────────────────────────────────────────
# Alignment: find changed regions
# ─────────────────────────────────────────────────────────────────────────────

def align_and_find_spans(
    unnorm: str,
    model_out: str
) -> List[Tuple[int, int, str]]:
    unnorm_tok = _tokenize(unnorm)
    model_tok = _tokenize(model_out)

    un_words = _tokens_only(unnorm_tok)
    mo_words = _tokens_only(model_tok)

    matcher = difflib.SequenceMatcher(None, un_words, mo_words, autojunk=False)

    spans = []
    pending_un_start = None
    pending_un_end = None
    pending_mo_parts = []

    def flush():
        if pending_un_start is not None and pending_mo_parts:
            replacement = " ".join(pending_mo_parts)
            spans.append((pending_un_start, pending_un_end, replacement))

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            flush()
            pending_un_start = None
            pending_un_end = None
            pending_mo_parts = []
        else:
            if i1 < i2:
                first_start = unnorm_tok[i1][1]
                last_tok, last_start = unnorm_tok[i2 - 1]
                last_end = last_start + len(last_tok)

                if pending_un_start is None:
                    pending_un_start = first_start
                pending_un_end = last_end
            pending_mo_parts.extend(mo_words[j1:j2])

    flush()
    return spans


# ─────────────────────────────────────────────────────────────────────────────
# Vote spans across models
# ─────────────────────────────────────────────────────────────────────────────

def vote_spans(
    unnorm: str,
    model_outputs: Dict[str, str],
) -> List[Dict]:
    range_to_models: Dict[Tuple[int, int], Dict[str, str]] = {}

    for model_name, model_out in model_outputs.items():
        if not model_out or not model_out.strip():
            continue
        for start, end, replacement in align_and_find_spans(unnorm, model_out):
            key = (start, end)
            if key not in range_to_models:
                range_to_models[key] = {}
            range_to_models[key][model_name] = replacement

    spans = []
    for (start, end), model_map in range_to_models.items():
        unnorm_span = unnorm[start:end]
        if not unnorm_span.strip():
            continue

        replacements = list(model_map.values())
        consensus = len(set(replacements)) == 1 and len(model_map) == len(model_outputs)

        spans.append({
            "unnorm_span": unnorm_span,
            "start": start,
            "end": end,
            "model_spans": model_map,
            "vote_count": len(model_map),
            "consensus": consensus,
        })

    spans.sort(key=lambda s: s["start"])
    return _remove_overlaps(spans)


def _remove_overlaps(spans: List[Dict]) -> List[Dict]:
    result = []
    used_end = -1
    for sp in spans:
        if sp["start"] >= used_end:
            result.append(sp)
            used_end = sp["end"]
        else:
            last = result[-1]
            if sp["vote_count"] > last["vote_count"] or (
                sp["vote_count"] == last["vote_count"] and
                (sp["end"] - sp["start"]) > (last["end"] - last["start"])
            ):
                result[-1] = sp
                used_end = sp["end"]
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Keys that are never treated as model outputs
# ─────────────────────────────────────────────────────────────────────────────

EXCLUDE_KEYS = {
    "unnorm", "norm", "tagged_ip", "tagged_op", "id",
    "cer_hints_ib", "cer_mt5", "cer_rb", "cer_indic_bart",
    "cer_gemma_1b", "cer_mt5_large", "wer_hints_ib",
    "wer_mt5", "wer_rb", "wer_indic_bart", "wer_gemma_1b",
    "wer_mt5_large", "gemma_1b", "mt5_large", "llama_1b",
    "cer_llama_1b", "wer_lama_1b",
}


def extract_model_outputs(raw: Dict) -> Dict[str, str]:
    """Pull only model-output keys from a raw record dict."""
    return {
        k: v.strip()
        for k, v in raw.items()
        if k not in EXCLUDE_KEYS and isinstance(v, str) and len(v) > 10
    }


# ─────────────────────────────────────────────────────────────────────────────
# Dataset - Dynamically detects ALL model keys  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

class SpanIdentificationDataset_up(Dataset):
    def __init__(self, file_path: str, tokenizer=None, max_length: int = 512):
        self.file_path = file_path
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data: List[Dict] = []
        self.model_keys: List[str] = []
        self._load()

    def _load(self):
        with open(self.file_path, encoding="utf-8") as f:
            for line_num, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                try:
                    raw = json.loads(line)
                except json.JSONDecodeError as e:
                    print(f"[Line {line_num}] JSON error: {e}")
                    continue

                unnorm = raw.get("unnorm", "").strip()
                norm = raw.get("norm", "").strip()

                model_outputs = extract_model_outputs(raw)
                for k in model_outputs:
                    if k not in self.model_keys:
                        self.model_keys.append(k)

                spans = vote_spans(unnorm, model_outputs)

                self.data.append({
                    "unnorm": unnorm,
                    "norm": norm,
                    "model_outputs": model_outputs,
                    "spans": spans,
                    "raw": raw,
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "unnorm_text": item["unnorm"],
            "norm_text": item["norm"],
            "model_outputs": item["model_outputs"],
            "spans": item["spans"],
            "num_spans": len(item["spans"]),
        }

    def get_model_keys(self) -> List[str]:
        return self.model_keys


# ─────────────────────────────────────────────────────────────────────────────
# Batch builder for classifier
# ─────────────────────────────────────────────────────────────────────────────

def build_batch(
    unnorm: str,
    spans: List[Dict],
    model_outputs: Dict[str, str],
) -> Dict:
    all_spans = []
    for sp in spans:
        ms = sp["model_spans"]
        span_entry = {
            "raw_span": sp["unnorm_span"],
        }
        for model_name, output in model_outputs.items():
            span_entry[f"{model_name}_output"] = ms.get(model_name, output)
        all_spans.append(span_entry)

    return {
        "sentences": [unnorm],
        "all_spans": all_spans,
        "span_to_sentence_idx": [0] * len(all_spans),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Classifier
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def classify_spans(
    model,
    batch: Dict,
    tokenizer,
    device: torch.device,
    model_keys: List[str],
) -> List[Dict]:
    model.eval()
    batch_dev = {
        k: (v.to(device) if torch.is_tensor(v) else v)
        for k, v in batch.items()
    }

    outputs = model(batch_dev, tokenizer)
    logits = outputs["logits"]
    probs = F.softmax(logits, dim=1)
    preds = torch.argmax(logits, dim=1).cpu()

    results = []
    for i, span_entry in enumerate(batch["all_spans"]):
        pred_idx = preds[i].item()
        pred_name = model_keys[pred_idx] if pred_idx < len(model_keys) else model_keys[0]
        norm_span = span_entry.get(f"{pred_name}_output", "")

        probs_dict = {}
        for idx, key in enumerate(model_keys):
            if idx < probs.shape[1]:
                probs_dict[key] = round(probs[i, idx].item(), 4)

        results.append({
            "model_picked": pred_name,
            "normalised_span": norm_span,
            "confidence": round(probs[i, pred_idx].item(), 4),
            "probabilities": probs_dict,
        })
    return results


# ─────────────────────────────────────────────────────────────────────────────
# Reconstruction
# ─────────────────────────────────────────────────────────────────────────────

def reconstruct(
    unnorm: str,
    spans: List[Dict],
    preds: List[Dict],
) -> str:
    parts = []
    cursor = 0

    for sp, pred in zip(spans, preds):
        start = sp["start"]
        end = sp["end"]
        norm = pred["normalised_span"].strip()

        if start < cursor:
            continue

        parts.append(unnorm[cursor:start])
        parts.append(norm)
        cursor = end

    parts.append(unnorm[cursor:])
    result = "".join(parts)
    result = re.sub(r" +", " ", result).strip()
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Full per-sentence inference  (unchanged internally)
# ─────────────────────────────────────────────────────────────────────────────

def infer_sentence(
    model,
    tokenizer,
    device: torch.device,
    item: Dict,
    model_keys: List[str],
) -> Dict:
    unnorm = item["unnorm"]
    norm = item["norm"]
    spans = item["spans"]
    mo = item["model_outputs"]
    raw = item.get("raw", {})

    if not spans:
        result = dict(raw)
        result["lm_span_selector"] = unnorm
        result["spans_info"] = []
        return result

    batch = build_batch(unnorm, spans, mo)
    preds = classify_spans(model, batch, tokenizer, device, model_keys)
    recon = reconstruct(unnorm, spans, preds)

    result = dict(raw)
    result["lm_span_selector"] = recon

    enriched = []
    for sp, pred in zip(spans, preds):
        enriched.append({
            "unnorm_span": sp["unnorm_span"],
            "start_char": sp["start"],
            "end_char": sp["end"],
            "model_picked": pred["model_picked"],
            "normalised_span": pred["normalised_span"],
            "confidence": pred["confidence"],
            "probabilities": pred["probabilities"],
            "vote_count": sp["vote_count"],
            "consensus": sp["consensus"],
        })
    result["spans_info"] = enriched

    return result


# ─────────────────────────────────────────────────────────────────────────────
# NEW: infer from a single raw dict
# ─────────────────────────────────────────────────────────────────────────────

def infer_single(
    model,
    tokenizer,
    device: torch.device,
    raw: Dict,
    model_keys: List[str],
) -> Dict:
    """
    Run the full pipeline on a single input dict.

    The dict must contain:
      - "unnorm"  : the unnormalised sentence (str)
      - one or more model-output keys (str values, len > 10)
        e.g. {"indicbart_op": "...", "llama_op": "...", "mt5_op": "..."}

    All other keys (norm, id, tagged_ip, …) are preserved as-is.
    Returns the original dict plus "lm_span_selector" and "spans_info".
    """
    unnorm        = raw.get("unnorm", "").strip()
    norm          = raw.get("norm", "").strip()
    model_outputs = extract_model_outputs(raw)

    # Update model_keys with any new keys seen in this record
    for k in model_outputs:
        if k not in model_keys:
            model_keys.append(k)

    spans = vote_spans(unnorm, model_outputs)

    item = {
        "unnorm":        unnorm,
        "norm":          norm,
        "model_outputs": model_outputs,
        "spans":         spans,
        "raw":           raw,
    }
    return infer_sentence(model, tokenizer, device, item, model_keys)


# ─────────────────────────────────────────────────────────────────────────────
# run_inference  (dataset-level loop — unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def run_inference(
    model,
    tokenizer,
    device: torch.device,
    dataset: SpanIdentificationDataset_up,
    output_path: str,
    show_progress: bool = True,
) -> List[Dict]:
    results = []
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    total = len(dataset)
    model_keys = dataset.get_model_keys()

    model_counts = {k: 0 for k in model_keys}

    print(f"\nRunning inference on {total} sentences → {output_path}")
    print(f"Models found: {model_keys}\n")

    with open(out_path, "w", encoding="utf-8") as fout:
        for idx, item in enumerate(dataset.data):
            result = infer_sentence(model, tokenizer, device, item, model_keys)
            results.append(result)
            fout.write(json.dumps(result, ensure_ascii=False) + "\n")

            for sp in result.get("spans_info", []):
                model_counts[sp["model_picked"]] += 1

            if show_progress and (idx + 1) % 100 == 0:
                print(f"  [{idx+1}/{total}]")

    total_spans = sum(model_counts.values())
    print(f"Done. {total} sentences, {total_spans} spans processed.")
    if total_spans:
        print("  Model selection:")
        for m, c in sorted(model_counts.items(), key=lambda x: -x[1]):
            print(f"    {m:15s}: {c:5d} ({c/total_spans*100:.1f}%)")

    return results


# ─────────────────────────────────────────────────────────────────────────────
# Usage
# ─────────────────────────────────────────────────────────────────────────────



In [47]:
model_keys = ["rb", "mt5", "hints_ib"]
test_dict = {
    "unnorm": "मुझे २,५०० रुपये चाहिए, तापमान २५.५°C है, समय २:४५ बजे है, और ३/४ लोग आएंगे",

    # RB: Strict number-to-word conversion
    "rb": "मुझे दो हजार पांच सौ रुपये चाहिए, तापमान पच्चीस दशमलव पांच डिग्री सेल्सियस है, समय दो बजकर पैंतालीस मिनट पर है, और तीन चौथाई लोग आएंगे",

    # MT5: Natural spoken with some shortcuts
    "mt5": "मुझे ढाई हजार रुपये चाहिए, तापमान पच्चीस-पांच डिग्री है, समय पौने तीन बजे है, और तीन चौथाई लोग आएंगे",

    # Hints_IB: Highly informal/colloquial
    "hints_ib": "मुझे पच्चीस सौ रु चाहिए, ताप २५.५° है, समय सवा तीन बजे है, और तिहाई लोग आएंगे",

    "id": "hard_test_002",
    "norm": "मुझे ढाई हजार रुपये चाहिए, तापमान पच्चीस दशमलव पांच डिग्री सेल्सियस है, समय पौने तीन बजे है, और तीन चौथाई लोग आएंगे"
}
    # ── 3. Run inference ──
result = infer_single(
        model=model_span,
        tokenizer=tokenizer,
        raw=test_dict,
        device=torch.device("cpu"),
        model_keys=model_keys
    )

In [48]:
result

{'unnorm': 'मुझे २,५०० रुपये चाहिए, तापमान २५.५°C है, समय २:४५ बजे है, और ३/४ लोग आएंगे',
 'rb': 'मुझे दो हजार पांच सौ रुपये चाहिए, तापमान पच्चीस दशमलव पांच डिग्री सेल्सियस है, समय दो बजकर पैंतालीस मिनट पर है, और तीन चौथाई लोग आएंगे',
 'mt5': 'मुझे ढाई हजार रुपये चाहिए, तापमान पच्चीस-पांच डिग्री है, समय पौने तीन बजे है, और तीन चौथाई लोग आएंगे',
 'hints_ib': 'मुझे पच्चीस सौ रु चाहिए, ताप २५.५° है, समय सवा तीन बजे है, और तिहाई लोग आएंगे',
 'id': 'hard_test_002',
 'norm': 'मुझे ढाई हजार रुपये चाहिए, तापमान पच्चीस दशमलव पांच डिग्री सेल्सियस है, समय पौने तीन बजे है, और तीन चौथाई लोग आएंगे',
 'lm_span_selector': 'मुझे ढाई हजार रुपये चाहिए, तापमान पच्चीस दशमलव पांच डिग्री सेल्सियस है, समय पौने तीन बजे है, और तिहाई लोग आएंगे',
 'spans_info': [{'unnorm_span': '२,५००',
   'start_char': 5,
   'end_char': 10,
   'model_picked': 'mt5',
   'normalised_span': 'ढाई हजार',
   'confidence': 0.6258,
   'probabilities': {'rb': 0.2219, 'mt5': 0.6258, 'hints_ib': 0.1523},
   'vote_count': 2,
   'consensus':